In [8]:
import pandas as pd

file_path = "SHELAR AGRO BOOKING FINAL.xlsx"

excel = pd.ExcelFile(file_path)

master = []

for sheet in excel.sheet_names:

    print(f"Processing {sheet}")

    # Skip the title rows
    df = pd.read_excel(
        file_path,
        sheet_name=sheet,
        header=None,
        skiprows=6,
        usecols="A:G"
    )

    # Assign proper column names
    df.columns = [
        "Date",
        "Room_1",
        "Room_2",
        "Room_3",
        "Room_4",
        "Room_5",
        "Room_6"
    ]

    # Convert dates
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    # Remove blank dates
    df = df[df["Date"].notna()]

    # Convert wide → long
    long_df = df.melt(
        id_vars="Date",
        var_name="Room_No",
        value_name="Guest_Name"
    )

    # Room number
    long_df["Room_No"] = (
        long_df["Room_No"]
        .str.replace("Room_", "", regex=False)
        .astype(int)
    )

    # Clean guest names
    long_df["Guest_Name"] = (
        long_df["Guest_Name"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    # Treat these as vacant
    long_df.loc[
        long_df["Guest_Name"].isin(
            ["", "0", "0.0", "nan", "None"]
        ),
        "Guest_Name"
    ] = pd.NA

    # Status
    long_df["Status"] = long_df["Guest_Name"].apply(
        lambda x: "Occupied" if pd.notna(x) else "Vacant"
    )

    # Date features
    long_df["Year"] = long_df["Date"].dt.year
    long_df["Month"] = long_df["Date"].dt.month_name()
    long_df["Day"] = long_df["Date"].dt.day
    long_df["Weekday"] = long_df["Date"].dt.day_name()

    master.append(long_df)

hotel_df = pd.concat(master, ignore_index=True)

hotel_df = hotel_df[
    [
        "Date",
        "Year",
        "Month",
        "Day",
        "Weekday",
        "Room_No",
        "Guest_Name",
        "Status",
    ]
]

hotel_df.to_csv("Hotel_Booking_Normalized.csv", index=False)

print(hotel_df.head())
print("\nRows:", len(hotel_df))
print("\nUnique Rooms:", hotel_df["Room_No"].unique())

Processing July-21
Processing August-21
Processing September 21
Processing October 21
Processing Nov 21
Processing Dec 21
Processing Jan 22
Processing Feb 22
Processing March 22
Processing Apr 22
Processing May 22
Processing June 22
Processing July 22
Processing Aug 22
Processing Sep22
Processing Oct 22
Processing Nov 22
Processing Dec 22
Processing Jan 23
Processing Feb 23
Processing Mar 23
Processing Apr 23
Processing May 23
Processing Jun 23
Processing Jul 23
Processing Aug 23
Processing Sep 23
Processing Oct 23
Processing Nov 23
Processing Dec 23
        Date  Year Month  Day   Weekday  Room_No       Guest_Name    Status
0 2021-07-08  2021  July    8  Thursday        1             <NA>    Vacant
1 2021-07-09  2021  July    9    Friday        1             <NA>    Vacant
2 2021-07-10  2021  July   10  Saturday        1  Mayuresh Kawale  Occupied
3 2021-07-11  2021  July   11    Sunday        1  Mayuresh Kawale  Occupied
4 2021-07-12  2021  July   12    Monday        1             <N

In [12]:
import pandas as pd

file_path = "Shelar Booking_2024.xlsx"

excel = pd.ExcelFile(file_path)

print(excel.sheet_names)

['Jan 24', 'Feb 24', 'March 24', 'April 24', 'May 24', 'June 24', 'July 24', 'Aug 24', 'Sep 24', 'Oct 24', 'Nov 24', 'Dec 24', 'Jan 25', 'Feb 25', 'March 25', 'April 25', 'May 25', 'June 25', 'July 25', 'Aug 25', 'Sept 25', 'Oct 25', 'Nov 25', 'Dec 25', 'Jan 26', 'Feb 26', 'March 26', 'April 26', 'May 26', 'June 26', 'July 26', 'Aug 26', 'Sept 26', 'Oct 26']


In [13]:
df = pd.read_excel("Shelar Booking_2024.xlsx", header=None)
print(df.shape)
print(df.head(15))

(33, 7)
                      0                  1                  2  \
0                   NaN                NaN                NaN   
1                  Date                  1                  2   
2   2024-01-01 00:00:00                NaN                NaN   
3   2024-01-02 00:00:00       Mayuri Bamne       Mayuri Bamne   
4   2024-01-03 00:00:00                NaN                NaN   
5   2024-01-04 00:00:00                NaN                NaN   
6   2024-01-05 00:00:00         Dev Gajjar     Vasant Jadhav    
7   2024-01-06 00:00:00         Dev Gajjar      Nikhil Landge   
8   2024-01-07 00:00:00                NaN      Nikhil Landge   
9   2024-01-08 00:00:00                NaN                NaN   
10  2024-01-09 00:00:00         Amit Dodke                NaN   
11  2024-01-10 00:00:00                NaN                NaN   
12  2024-01-11 00:00:00                NaN                NaN   
13  2024-01-12 00:00:00  Ad Kishore Hazare  Ad Kishore Hazare   
14  2024-01-13 00

In [16]:
import pandas as pd

# ==========================
# File Path
# ==========================
file_path = "Shelar Booking_2024.xlsx"

excel = pd.ExcelFile(file_path)

master = []

for sheet in excel.sheet_names:

    print(f"Processing: {sheet}")

    # Read sheet
    df = pd.read_excel(
        file_path,
        sheet_name=sheet,
        header=1
    )

    # Skip empty sheets
    if df.empty:
        print(f"Skipping {sheet} (empty)")
        continue

    # Some sheets may have fewer columns
    if len(df.columns) < 7:
        print(f"Skipping {sheet} (only {len(df.columns)} columns)")
        continue

    # Keep only first 7 columns
    df = df.iloc[:, :7].copy()

    # Rename columns
    df.columns = [
        "Date",
        "Room_1",
        "Room_2",
        "Room_3",
        "Room_4",
        "Room_5",
        "Room_6"
    ]

    # Convert Date
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    # Remove invalid dates
    df = df[df["Date"].notna()]

    # Convert Wide → Long
    long_df = df.melt(
        id_vars="Date",
        var_name="Room_No",
        value_name="Guest_Name"
    )

    # Extract room number
    long_df["Room_No"] = (
        long_df["Room_No"]
        .str.extract(r'(\d+)')
        .astype(int)
    )

    # Clean guest names
    long_df["Guest_Name"] = (
        long_df["Guest_Name"]
        .astype(str)
        .str.strip()
    )

    # Treat these as vacant
    long_df["Guest_Name"] = long_df["Guest_Name"].replace(
        ["", "nan", "None", "0", "0.0"],
        pd.NA
    )

    # Status
    long_df["Status"] = long_df["Guest_Name"].apply(
        lambda x: "Occupied" if pd.notna(x) else "Vacant"
    )

    # Date Features
    long_df["Year"] = long_df["Date"].dt.year
    long_df["Month"] = long_df["Date"].dt.month_name()
    long_df["Day"] = long_df["Date"].dt.day
    long_df["Weekday"] = long_df["Date"].dt.day_name()

    master.append(long_df)

# Combine all sheets
hotel_df = pd.concat(master, ignore_index=True)

# Arrange columns
hotel_df = hotel_df[
    [
        "Date",
        "Year",
        "Month",
        "Day",
        "Weekday",
        "Room_No",
        "Guest_Name",
        "Status",
    ]
]

# Sort data
hotel_df = hotel_df.sort_values(
    ["Date", "Room_No"]
).reset_index(drop=True)

hotel_df = hotel_df[
    hotel_df["Date"] <= "2026-06-30"
]
# Save CSV
hotel_df.to_csv("Hotel_Booking_Normalized_2024.csv", index=False)

print("\nDone!")
print(hotel_df.head(12))
print(f"\nTotal Records: {len(hotel_df)}")
print(f"Unique Rooms: {hotel_df['Room_No'].unique()}")

Processing: Jan 24
Processing: Feb 24
Processing: March 24
Processing: April 24
Processing: May 24
Processing: June 24
Processing: July 24
Processing: Aug 24
Processing: Sep 24
Processing: Oct 24
Processing: Nov 24
Processing: Dec 24
Processing: Jan 25
Processing: Feb 25
Processing: March 25
Processing: April 25
Processing: May 25
Processing: June 25
Processing: July 25
Processing: Aug 25
Processing: Sept 25
Processing: Oct 25
Processing: Nov 25
Processing: Dec 25
Processing: Jan 26
Processing: Feb 26
Processing: March 26
Processing: April 26
Processing: May 26
Processing: June 26
Processing: July 26
Processing: Aug 26
Skipping Aug 26 (empty)
Processing: Sept 26
Skipping Sept 26 (empty)
Processing: Oct 26
Skipping Oct 26 (empty)

Done!
         Date  Year    Month  Day  Weekday  Room_No    Guest_Name    Status
0  2024-01-01  2024  January    1   Monday        1          <NA>    Vacant
1  2024-01-01  2024  January    1   Monday        2          <NA>    Vacant
2  2024-01-01  2024  Janua

In [17]:
import pandas as pd

df1 = pd.read_csv(
    "Hotel_Booking_Normalized.csv",
    parse_dates=["Date"]
)

df2 = pd.read_csv(
    "Hotel_Booking_Normalized_2024.csv",
    parse_dates=["Date"]
)

master_df = pd.concat(
    [df1, df2],
    ignore_index=True
)

master_df = master_df.sort_values(
    ["Date", "Room_No"]
).reset_index(drop=True)

master_df.to_csv(
    "Hotel_Booking_Master.csv",
    index=False
)

print(master_df.head())
print(master_df.tail())
print(master_df.shape)

        Date  Year Month  Day   Weekday  Room_No Guest_Name  Status
0 2021-07-08  2021  July    8  Thursday        1        NaN  Vacant
1 2021-07-08  2021  July    8  Thursday        2        NaN  Vacant
2 2021-07-08  2021  July    8  Thursday        3        NaN  Vacant
3 2021-07-08  2021  July    8  Thursday        4        NaN  Vacant
4 2021-07-08  2021  July    8  Thursday        5        NaN  Vacant
            Date  Year Month  Day  Weekday  Room_No Guest_Name  Status
10015 2026-06-30  2026  June   30  Tuesday        2        NaN  Vacant
10016 2026-06-30  2026  June   30  Tuesday        3        NaN  Vacant
10017 2026-06-30  2026  June   30  Tuesday        4        NaN  Vacant
10018 2026-06-30  2026  June   30  Tuesday        5        NaN  Vacant
10019 2026-06-30  2026  June   30  Tuesday        6        NaN  Vacant
(10020, 8)
